## Once-for-All (OFA)

**Ý tưởng:**  
Huấn luyện một **supernet** duy nhất có khả năng sinh ra hàng triệu mô hình con (sub-networks) phù hợp với nhiều thiết bị khác nhau.  
Thay vì train lại cho từng trường hợp, OFA chỉ train một lần, sau đó chọn mô hình con tối ưu theo constraint (latency, FLOPs, accuracy).


### 1. Progressive Shrinking
- Huấn luyện supernet với chiến lược thu nhỏ dần:
  - Kernel size: $7 \to 5 \to 3$  
  - Depth: giảm số layer  
  - Width: giảm số kênh  
  - Resolution: giảm kích thước input  
- Giúp supernet bao phủ được nhiều cấu hình khác nhau.


In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torchvision
import torchvision.transforms as T
from torch.utils.data import DataLoader
import random

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 128
NUM_CLASSES = 10

mean = (0.4914, 0.4822, 0.4465)
std = (0.2470, 0.2435, 0.2616)
train_tf = T.Compose([T.RandomCrop(32, 4), T.RandomHorizontalFlip(), T.ToTensor(), T.Normalize(mean, std)])
test_tf = T.Compose([T.ToTensor(), T.Normalize(mean, std)])

train_set = torchvision.datasets.CIFAR10(root="./data", train=True, download=True, transform=train_tf)
test_set = torchvision.datasets.CIFAR10(root="./data", train=False, download=True, transform=test_tf)
train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
test_loader = DataLoader(test_set, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

/home/tam/miniconda3/envs/first-env/lib/python3.13/site-packages/torch/cuda/__init__.py:182: UserWarning: CUDA initialization: CUDA unknown error - this may be due to an incorrectly set up environment, e.g. changing env variable CUDA_VISIBLE_DEVICES after program start. Setting the available devices to be zero. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:109.)
  return torch._C._cuda_getDeviceCount() > 0


### 2. Weight Sharing
- Mọi mô hình con đều chia sẻ trọng số từ supernet.  
- Không cần huấn luyện lại từng mô hình con → tiết kiệm chi phí.


In [2]:
class ElasticConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, kernel_sizes=[3,5,7], stride=1):
        super().__init__()
        self.kernel_sizes = kernel_sizes
        self.convs = nn.ModuleDict({
            str(k): nn.Conv2d(in_ch, out_ch, k, stride=stride, padding=k//2)
            for k in kernel_sizes
        })
        self.active_kernel = str(kernel_sizes[0])
        self.bn = nn.BatchNorm2d(out_ch)
        self.relu = nn.ReLU(inplace=True)
    def set_kernel(self, k):
        self.active_kernel = str(k)
    def forward(self, x):
        x = self.convs[self.active_kernel](x)
        x = self.bn(x)
        return self.relu(x)



### 3. Elastic Architecture
- Supernet hỗ trợ nhiều kiến trúc co giãn:
  - Elastic Depth  
  - Elastic Width  
  - Elastic Kernel Size  
  - Elastic Resolution


In [5]:
class SimpleOFA(nn.Module):
    def __init__(self, num_classes=10, widths=[32, 64, 128], kernel_sizes=[3,5,7]):
        super().__init__()
        self.widths = widths
        self.kernel_sizes = kernel_sizes

        self.stem = nn.Sequential(
            nn.Conv2d(3, widths[0], kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(widths[0]),
            nn.ReLU(inplace=True)
        )

        self.blocks = nn.ModuleList([
            ElasticConvBlock(widths[i], widths[i+1], kernel_sizes)
            for i in range(len(widths)-1)
        ])

        # Adapter để map mọi depth -> 128 channels
        self.adapters = nn.ModuleDict({
            str(c): nn.Conv2d(c, widths[-1], kernel_size=1)
            for c in widths
        })

        self.classifier = nn.Linear(widths[-1], num_classes)
        self.active_depth = len(self.blocks)

    def set_active_subnet(self, depth=None, kernel=None):
        if depth is not None:
            self.active_depth = depth
        if kernel is not None:
            for block in self.blocks:
                block.set_kernel(kernel)

    def forward(self, x):
        x = self.stem(x)
        for i, block in enumerate(self.blocks):
            if i >= self.active_depth:
                break
            x = block(x)
        # map về 128 channels trước khi avg pool
        c = x.size(1)
        x = self.adapters[str(c)](x)
        x = F.adaptive_avg_pool2d(x, 1).flatten(1)
        return self.classifier(x)

supernet = SimpleOFA().to(DEVICE)

### 4. Evolutionary Search with Constraints
- Dùng giải thuật tiến hóa để tìm mô hình con tối ưu.  
- Constraint: Latency, FLOPs, memory footprint, accuracy.  
- Latency được đo trực tiếp trên thiết bị target.


In [7]:
# %% [markdown]
# ### 3. Once-for-All Training (Progressive Shrinking)

EPOCHS = 10
optimizer = optim.SGD(supernet.parameters(), lr=0.1, momentum=0.9, weight_decay=5e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

def train_one_epoch(model, loader, optimizer):
    model.train()
    total, correct = 0, 0
    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        # Progressive shrinking: random depth & kernel mỗi batch
        depth = random.choice([1,2,3])
        kernel = random.choice([3,5,7])
        model.set_active_subnet(depth=depth, kernel=kernel)
        optimizer.zero_grad()
        out = model(x)
        loss = F.cross_entropy(out, y)
        loss.backward()
        optimizer.step()
        correct += (out.argmax(1) == y).sum().item()
        total += y.size(0)
    return correct / total

for epoch in range(EPOCHS):
    acc = train_one_epoch(supernet, train_loader, optimizer)
    scheduler.step()
    print(f"Epoch {epoch+1}/{EPOCHS} - Train acc: {acc:.4f}")

Epoch 1/10 - Train acc: 0.4124
Epoch 2/10 - Train acc: 0.4413
Epoch 3/10 - Train acc: 0.4743
Epoch 4/10 - Train acc: 0.4900
Epoch 5/10 - Train acc: 0.5200
Epoch 6/10 - Train acc: 0.5468
Epoch 7/10 - Train acc: 0.5647
Epoch 8/10 - Train acc: 0.5849
Epoch 9/10 - Train acc: 0.6013
Epoch 10/10 - Train acc: 0.6140


In [8]:
# %% [markdown]
# ### 4. Evolutionary Search (Chọn subnet tối ưu theo constraint)

subnet_settings = [
    {"name": "small", "depth": 1, "kernel": 3},
    {"name": "medium", "depth": 2, "kernel": 5},
    {"name": "large", "depth": 3, "kernel": 7},
]

def evaluate(model, loader):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            out = model(x)
            correct += (out.argmax(1) == y).sum().item()
            total += y.size(0)
    return correct / total

for cfg in subnet_settings:
    supernet.set_active_subnet(depth=cfg["depth"], kernel=cfg["kernel"])
    acc = evaluate(supernet, test_loader)
    print(f"Subnet {cfg['name']}: depth={cfg['depth']}, kernel={cfg['kernel']} | Test acc = {acc:.4f}")

Subnet small: depth=1, kernel=3 | Test acc = 0.3605
Subnet medium: depth=2, kernel=5 | Test acc = 0.4154
Subnet large: depth=3, kernel=7 | Test acc = 0.5064


### 5. Once-for-All Training
- Toàn bộ supernet được huấn luyện **một lần duy nhất**.  
- Sau đó có thể trích xuất nhanh các sub-networks phù hợp cho từng thiết bị.


### Metric
- **Accuracy**: độ chính xác phân loại.  
- **Latency**: độ trễ đo trên thiết bị thật.  
- **FLOPs**: số phép toán.  
- **Model size**: dung lượng mô hình.  



In [11]:
# %% [markdown]
# ### 5. Metric: Accuracy, Latency, FLOPs, Model size

def count_params(model):
    return sum(p.numel() for p in model.parameters())

def measure_latency(model, input_size=(1,3,32,32), repeat=30, warmup=10):
    model.eval()
    x = torch.randn(*input_size, device=DEVICE)
    with torch.no_grad():
        for _ in range(warmup):
            _ = model(x)
        torch.cuda.synchronize() if torch.cuda.is_available() else None
        import time
        times = []
        for _ in range(repeat):
            t0 = time.time()
            _ = model(x)
            torch.cuda.synchronize() if torch.cuda.is_available() else None
            t1 = time.time()
            times.append(t1-t0)
    return 1000 * sum(times)/len(times)  # ms

for cfg in subnet_settings:
    supernet.set_active_subnet(depth=cfg["depth"], kernel=cfg["kernel"])
    params = count_params(supernet)
    latency = measure_latency(supernet)
    print(f"Subnet {cfg['name']}: Params = {params}, Latency = {latency:.2f} ms/img")
    # FLOPs: dùng thop nếu muốn, ví dụ:
    from thop import profile
    macs, flops = profile(supernet, inputs=(torch.randn(1,3,32,32).to(DEVICE),), verbose=False)
    print(f"FLOPs: {flops/1e6:.2f} MFLOPs")

Subnet small: Params = 882154, Latency = 0.51 ms/img
FLOPs: 0.03 MFLOPs
Subnet medium: Params = 882154, Latency = 2.84 ms/img
FLOPs: 0.28 MFLOPs
Subnet large: Params = 882154, Latency = 6.86 ms/img
FLOPs: 0.52 MFLOPs
